# RegLens · Validation — AUROC on the matched MPRA benchmark

Runs the **credibility linchpin**: does the pretrained ChromBPNet model's variant score
(`|Δ log-counts|`) rank **functional** regulatory variants above **non-functional** ones
— on the *hard, matched* comparison — and does it beat CADD?

### Honest expectations (read first)
- Negatives are **matched within-element** (Kircher satMutMPRA): the model must separate
  functional from non-functional variants *in the same active regions*. This is HARD.
- **Do not expect 0.9.** A modest AUROC (~0.60–0.70) that **beats CADD** on this fair
  benchmark is a legitimate, strong, honest result. Report it straight either way —
  per-element and overall, with the class balance.
- This validates the **ENGINE** (the variant score). The **AGENT** (multi-agent
  interpretation) is a *separate* claim, validated by recovering rs1427407 / rs2814778
  mechanisms + the red-team catching artifacts. Don't conflate them.

### Two passes (don't let 33k variants become a Sunday-night surprise)
1. **Subset (BCL11A, ~1.5k variants)** → a real number fast + confirm the pipeline scales.
2. **Full 33k** on the GPU → per-element + overall AUROC.

**Requires a GPU runtime.**

In [ ]:
# --- GPU + install -----------------------------------------------------------
!nvidia-smi -L || echo 'NO GPU — switch Runtime to GPU.'
!pip -q install 'tensorflow>=2.12' pyfaidx typer numpy matplotlib
# Upload the RegLens repo zip (reglens_for_colab.zip) via the Files panel, then:
!unzip -oq /content/reglens_for_colab.zip -d /content/RegLens && pip -q install -e /content/RegLens
import os; os.chdir('/content/RegLens'); print('cwd', os.getcwd())

In [ ]:
# --- Genome + pretrained model ----------------------------------------------
import os
os.makedirs('/content/ref', exist_ok=True)
%cd /content/ref
# hg38 — try UCSC main + EU + GI mirrors (in case one host's DNS is flaky).
!for h in hgdownload.soe.ucsc.edu hgdownload-euro.soe.ucsc.edu hgdownload.gi.ucsc.edu; do \
   echo "trying $h"; wget -c "https://$h/goldenPath/hg38/bigZips/hg38.fa.gz" && break; done
# Fallback if all UCSC hosts fail DNS: Broad public hg38 on GCS (chr-named, ships .fai):
#   !gsutil -q cp gs://gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta hg38.fa
#   !gsutil -q cp gs://gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta.fai hg38.fa.fai
!gunzip -kf hg38.fa.gz
!ls -lh hg38.fa   # expect ~3.1 GB
!python -c "import pyfaidx; pyfaidx.Faidx('hg38.fa'); print('faidx OK')"
# ENCODE K562 ATAC ChromBPNet models (all 5 folds).
![ -f ENCFF984RAF.tar.gz ] || wget -q -c -O ENCFF984RAF.tar.gz https://www.encodeproject.org/files/ENCFF984RAF/@@download/ENCFF984RAF.tar.gz
!mkdir -p encode_models && tar -xzf ENCFF984RAF.tar.gz -C encode_models
import glob; folds=glob.glob('encode_models/**/*chrombpnet_nobias*.h5', recursive=True); print(len(folds),'folds')
%cd /content/RegLens

In [ ]:
# --- PASS 1: subset (BCL11A) — fast real number -----------------------------
from reglens.validation import load_labeled_variants, evaluate
from reglens.tools.chrombpnet_score import ChromBPNetScorer, load_backend
from reglens.report.render import render_validation
import time

bench = load_labeled_variants('data/benchmarks/kircher_mpra_grch38.tsv')
bcl11a = [v for v in bench if v.source == 'BCL11A']
print(f'BCL11A subset: {len(bcl11a)} variants, {sum(v.label for v in bcl11a)} positive')

# Fold + reverse-complement averaged backend over all folds.
scorer = ChromBPNetScorer(load_backend('/content/ref/encode_models', average_rc=True),
                          window_length=2114, model_name='K562-5fold+rc')
t0=time.time()
rep = evaluate(bcl11a, scorer, genome_path='/content/ref/hg38.fa', progress=True)
print(f'scored in {time.time()-t0:.0f}s')
print(render_validation(rep))

### CADD baseline (for the comparison)
1. Export the benchmark's variants to VCF and submit to the **CADD web service**
   (https://cadd.gs.washington.edu/, GRCh38 v1.6), download the scored `.tsv.gz`, gunzip.
2. Merge it onto the benchmark — the harness then reads the `cadd` column automatically:
```python
from reglens.validation.cadd import annotate_benchmark
annotate_benchmark('data/benchmarks/kircher_mpra_grch38.tsv', 'cadd_scores.tsv',
                   'data/benchmarks/kircher_mpra_grch38.cadd.tsv')
```
Then load `...cadd.tsv` instead. Model AUROC is reported with or without CADD.

In [ ]:
# --- PASS 2: full 33k (GPU) — per-element + overall --------------------------
# Heavy: 33k variants × (ref+alt) × fwd/RC × folds. Run on GPU; expect a while.
import time
t0=time.time()
full = evaluate_batched(bench, scorer, genome_path='/content/ref/hg38.fa', chunk_size=2000, progress=True)
print(f'scored {len(bench)} variants in {time.time()-t0:.0f}s')
print(render_validation(full))
import json; print(json.dumps(full.to_dict(), indent=2))

In [ ]:
# --- ROC money-shot figure (model vs CADD) ----------------------------------
from reglens.report.plot import plot_roc
plot_roc(full, '/content/roc_mpra.png')
from IPython.display import Image; Image('/content/roc_mpra.png')

In [ ]:
# --- Cell-type specificity + per-element figure -----------------------------
from reglens.validation.lineage import render_celltype_specificity
from reglens.report.plot import plot_per_element
print(render_celltype_specificity(full))
plot_per_element(full, '/content/per_element_auroc.png')
from IPython.display import Image; Image('/content/per_element_auroc.png')

## Reporting (honest)
- State **overall AUROC** and **per-element AUROC**, and the **class balance**
  (positive/negative counts) — `render_validation` prints all three.
- If model **beats CADD by a clear margin** on this matched benchmark → that's the headline.
- If it's **close**, report it straight. Matched within-element negatives is a hard task;
  a modest AUROC that edges CADD is a real, defensible result.
- Keep the engine claim (this AUROC) separate from the agent claim (mechanism recovery).